# FIRMS data from GEE to GCS

### Authentication

In [1]:
!earthengine authenticate

Authenticate: Limited support in Colab. Use ee.Authenticate() or --auth_mode=notebook instead.
To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=KH0nNLShIMFUF6KfqpBnZoHFagG_npjghEfeyA_G1BE&tc=5tMLwUOopasABQE5r04bLqLZ_2SeHZNCzTXBovHmycQ&cc=d_BtXAsCml1Bq93l8bNXCmkdIu6uk1BMKZ-Y6cOITM8

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AXEQxIAUVzm3hXfbiVQsdaMGetJhf9SL2LzJTPJ6k2cBhPnX0buu8t1FRCc

Successfully saved authorization token.


### Build the image, one for each day and export

In [ ]:
import ee
from datetime import datetime, timedelta
import time

# -----------------------------
# 1. Initialize Earth Engine
# -----------------------------
try:
    ee.Initialize(project='project-45996894-227e-417a-a7c')
except Exception:
    print('Earth Engine is not authenticated yet.')
    raise

# -----------------------------
# 2. Config
# -----------------------------
BUCKET = 'wildfire-detection-first'
PREFIX = 'firms_daily_geotiff'
START_DATE = '2025-09-25'
END_DATE = '2025-09-25'   # inclusive
SCALE = 1000
CRS = 'EPSG:4326'
MAX_PIXELS = 1e13
SLEEP_BETWEEN_TASKS = 0.5

# -----------------------------
# 3. California AOI
# -----------------------------
states = ee.FeatureCollection('TIGER/2018/States')
california = states.filter(ee.Filter.eq('NAME', 'California'))
aoi = california.geometry()

# -----------------------------
# 4. Helper to build one day image
# -----------------------------
def make_firms_day_image(day_str):
    day_start = ee.Date(day_str)
    day_end = day_start.advance(1, 'day')

    firms_col = (
        ee.ImageCollection('FIRMS')
        .filterBounds(aoi)
        .filterDate(day_start, day_end)
    )

    confidence = firms_col.select('confidence').max().rename('firms_confidence')
    t21 = firms_col.select('T21').max().rename('firms_t21')
    label = firms_col.select('T21').count().gt(0).rename('label').toFloat()

    img = (
        ee.Image.cat([confidence, t21, label])
        .clip(aoi)
        .toFloat()
        .set('date_str', day_str)
        .set('system:time_start', day_start.millis())
    )
    return img

# -----------------------------
# 5. Export one day
# -----------------------------
def make_export_task(day_str):
    img = make_firms_day_image(day_str)

    task = ee.batch.Export.image.toCloudStorage(
        image=img,
        description=f'firms_{day_str}',
        bucket=BUCKET,
        fileNamePrefix=f'{PREFIX}/{day_str}',
        region=aoi,
        scale=SCALE,
        crs=CRS,
        maxPixels=MAX_PIXELS,
        fileFormat='GeoTIFF'
    )
    return task

# -----------------------------
# 6. Launch daily tasks
# -----------------------------
start_dt = datetime.strptime(START_DATE, '%Y-%m-%d')
end_dt = datetime.strptime(END_DATE, '%Y-%m-%d')

tasks = []
current = start_dt

while current <= end_dt:
    day_str = current.strftime('%Y-%m-%d')
    task = make_export_task(day_str)
    task.start()
    tasks.append((day_str, task))
    print(f'Started task for {day_str}: {task.id}')
    time.sleep(SLEEP_BETWEEN_TASKS)
    current += timedelta(days=1)

print(f'\nStarted {len(tasks)} tasks.')

Started task for 2025-09-25: RUSMTIU4DHR2H2APBUB5ZMPD

Started 1 tasks.


### Check if job completed successfully

In [ ]:
import ee
ee.Initialize(project='project-45996894-227e-417a-a7c')

ops = ee.data.listOperations()

pending_states = {'PENDING', 'RUNNING', 'CANCELLING'}
pending = [op for op in ops if op.get('metadata', {}).get('state') in pending_states]

print('Total operations:', len(ops))
print('Pending/running:', len(pending))

Total operations: 4039
Pending/running: 1


### Min and Max date available

In [2]:
from google.cloud import storage
from datetime import datetime
from google.colab import auth

auth.authenticate_user()

BUCKET = 'wildfire-detection-first'
PREFIX = 'firms_daily_geotiff'

client = storage.Client()
bucket = client.get_bucket(BUCKET)

blobs = bucket.list_blobs(prefix=PREFIX)

object_count = 0
dates = []

for blob in blobs:
    # Exclude directory placeholders if they exist (blobs ending with '/').
    # Only consider actual files, which are expected to have dates in their names.
    if not blob.name.endswith('/'):
        object_count += 1
        # Extract date from blob name, assuming format like 'PREFIX/YYYY-MM-DD_...'
        try:
            # The date part is after PREFIX/ and before the next underscore or end of string
            date_str_start = len(PREFIX) + 1 # +1 for the slash
            date_str = blob.name[date_str_start:date_str_start + 10] # YYYY-MM-DD is 10 chars
            dates.append(datetime.strptime(date_str, '%Y-%m-%d'))
        except ValueError:
            print(f"Warning: Could not parse date from filename: {blob.name}")

print(f"Total objects in '{BUCKET}/{PREFIX}/': {object_count}")

if dates:
    min_date = min(dates)
    max_date = max(dates)
    print(f"Minimum date in filenames: {min_date.strftime('%Y-%m-%d')}")
    print(f"Maximum date in filenames: {max_date.strftime('%Y-%m-%d')}")
else:
    print("No dates found in filenames.")

Total objects in 'wildfire-detection-first/firms_daily_geotiff/': 3643
Minimum date in filenames: 2016-01-01
Maximum date in filenames: 2025-12-31


### Missing dates

In [3]:
from datetime import datetime, timedelta

# Define the expected start and end dates from the user's prompt
expected_start_date = datetime(2016, 1, 1)
expected_end_date = datetime(2025, 12, 31)

# Create a set of all expected dates in the range
expected_dates = set()
current_date = expected_start_date
while current_date <= expected_end_date:
    expected_dates.add(current_date)
    current_date += timedelta(days=1)

# Convert the dates found in GCS to a set for efficient comparison
found_dates_set = set(dates)

# Find missing dates
missing_dates = sorted(list(expected_dates - found_dates_set))

if missing_dates:
    print(f"Found {len(missing_dates)} missing dates between {expected_start_date.strftime('%Y-%m-%d')} and {expected_end_date.strftime('%Y-%m-%d')}:")
    for date in missing_dates:
        print(date.strftime('%Y-%m-%d'))
else:
    print("No missing dates found in the specified range.")

Found 11 missing dates between 2016-01-01 and 2025-12-31:
2017-03-25
2017-03-26
2017-03-27
2017-03-28
2017-03-29
2017-03-30
2017-03-31
2017-04-01
2017-04-02
2018-07-07
2018-07-08
